In [ ]:
# ==============================================================================
# SEÇÃO 1: IMPORTAÇÃO DAS BIBLIOTECAS
# ==============================================================================
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
# --- IMPORTAÇÃO PARA GRÁFICOS ---
import matplotlib.pyplot as plt

print("Bibliotecas importadas com sucesso.")


In [ ]:

# ==============================================================================
# SEÇÃO 2: CARREGAMENTO DOS DADOS
# ==============================================================================
print("\nCarregando os datasets (train, test, stores, features)...")
try:
    df_train = pd.read_csv('train.csv')
    df_test = pd.read_csv('test.csv')
    df_stores = pd.read_csv('stores.csv')
    df_features = pd.read_csv('features.csv')
    print("Datasets carregados com sucesso.")
except FileNotFoundError as e:
    print(f"Erro: Arquivo não encontrado. Verifique se os arquivos CSV estão na mesma pasta que o script. Detalhe: {e}")
    exit()


In [ ]:

# ==============================================================================
# SEÇÃO 3: MERGE E PRÉ-PROCESSAMENTO DOS DADOS
# ==============================================================================
print("\nIniciando o pré-processamento e unificação dos dados...")
df_features_stores = pd.merge(df_features, df_stores, on='Store', how='left')
df_train_completo = pd.merge(df_train, df_features_stores, on=['Store', 'Date', 'IsHoliday'], how='left')
df_test_completo = pd.merge(df_test, df_features_stores, on=['Store', 'Date', 'IsHoliday'], how='left')
df_train_completo.fillna(0, inplace=True)
df_test_completo.fillna(0, inplace=True)
print("Merge e tratamento de valores nulos concluídos para ambos os datasets.")


In [ ]:
# ==============================================================================
# SEÇÃO 4: ENGENHARIA DE ATRIBUTOS (FEATURE ENGINEERING)
# ==============================================================================
print("\nCriando novas features para ambos os datasets...")
def criar_features(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Ano'] = df['Date'].dt.year
    df['Mes'] = df['Date'].dt.month
    df['Dia'] = df['Date'].dt.day
    df['Semana_do_Ano'] = df['Date'].dt.isocalendar().week.astype(int)
    df['IsHoliday'] = df['IsHoliday'].astype(int)
    return df

df_train_final = criar_features(df_train_completo)
df_test_final = criar_features(df_test_completo)

df_full = pd.concat([df_train_final, df_test_final])
df_full = pd.get_dummies(df_full, columns=['Type'], prefix='Tipo')

df_train_final = df_full[df_full['Weekly_Sales'].notna()]
df_test_final = df_full[df_full['Weekly_Sales'].isna()]
print("Novas features criadas com sucesso.")

In [ ]:
# ==============================================================================
# SEÇÃO 5: CRIANDO UM CONJUNTO DE VALIDAÇÃO
# ==============================================================================
data_corte_validacao = '2012-01-01'
print(f"\nDividindo o dataset de treino em treino e validação (corte em {data_corte_validacao}).")
train_set = df_train_final[df_train_final['Date'] < data_corte_validacao]
validation_set = df_train_final[df_train_final['Date'] >= data_corte_validacao]

target = 'Weekly_Sales'
features = [col for col in df_train_final.columns if col not in [target, 'Date']]

X_train = train_set[features]
y_train = train_set[target]
X_val = validation_set[features]
y_val = validation_set[target]
print(f"Tamanho do conjunto de treino: {len(X_train)} linhas")
print(f"Tamanho do conjunto de validação: {len(X_val)} linhas")